# Exploración del Lakh MIDI Dataset

**Objetivo:** entender qué hay en los archivos MIDI antes de diseñar el modelo de key detection.

Antes de entrenar cualquier modelo neuronal, la regla de oro es **mirar los datos primero**. 
Los errores más caros en ML no son de arquitectura — son de no entender el dataset.

Este notebook responde:
1. ¿Cómo se ve un archivo MIDI en términos de notas y timing?
2. ¿Qué notas aparecen más en el dataset? ¿Hay sesgos obvios?
3. ¿El pitch class histogram es una buena representación para key detection?
4. ¿Cuántas notas por segundo hay típicamente? ¿Es un dataset activo o sparse?

**Setup:** primero corré `uv run python download_lakh.py` desde este directorio.

In [ ]:
import sys
from pathlib import Path

# Agregar utils al path para importar desde notebooks
repo_root = Path.cwd().parent
sys.path.insert(0, str(repo_root))

import numpy as np
import matplotlib.pyplot as plt
import pretty_midi

from utils.midi_utils import (
    load_midi,
    get_pitch_class_histogram,
    midi_to_note_sequence,
)

print("Imports OK")
print(f"pretty_midi version: {pretty_midi.__version__}")

## 1. Cargar archivos MIDI

Buscamos archivos `.mid` en el directorio `datasets/lakh/`. Si corriste `download_lakh.py` 
en modo muestra, habrá al menos 1 archivo sintético. Si corriste el download completo, 
habrá hasta 500.

**¿Por qué cargar 5-10 y no todos?** La exploración es visual — no tiene sentido cargar 
500 archivos para hacer gráficos. Usamos una muestra para entender la distribución, 
y luego procesamos el batch completo en el script de training.

In [ ]:
LAKH_DIR = Path("lakh")
MAX_FILES = 10  # Para exploración visual, 10 es suficiente

midi_files = sorted(LAKH_DIR.glob("*.mid"))[:MAX_FILES]

if not midi_files:
    print("No hay archivos MIDI en datasets/lakh/")
    print("Corré: uv run python datasets/download_lakh.py")
else:
    print(f"Encontrados: {len(midi_files)} archivos MIDI")
    for f in midi_files:
        print(f"  {f.name}")

In [ ]:
# Cargar todos los archivos encontrados
midis = []
loaded_names = []

for path in midi_files:
    try:
        midi = load_midi(path)
        midis.append(midi)
        loaded_names.append(path.name)
        print(f"OK  {path.name}  ({midi.get_end_time():.1f}s, {len(midi.instruments)} instrumentos)")
    except Exception as e:
        print(f"ERR {path.name}: {e}")

print(f"\nCargados exitosamente: {len(midis)}/{len(midi_files)}")

## 2. Piano Roll — visualización de notas en tiempo

El **piano roll** es la representación más intuitiva de un archivo MIDI: 
el eje X es tiempo, el eje Y es nota (60 = C4 = Do central), 
y cada rectángulo es una nota con su duración.

Qué buscamos:
- ¿El archivo tiene notas en un rango razonable (octavas 2-7, MIDI 24-96)?
- ¿Hay poliritmia? ¿Muchas notas simultáneas?
- ¿La densidad temporal es alta (jazz) o baja (balada lenta)?

Esto afecta las decisiones de arquitectura: un modelo para música densa 
necesita ventanas temporales más cortas que uno para música sparse.

In [ ]:
def plot_piano_roll(midi: pretty_midi.PrettyMIDI, title: str, max_time: float = 30.0):
    """Visualiza las primeras max_time segundos del piano roll."""
    fig, ax = plt.subplots(figsize=(14, 4))
    
    colors = plt.cm.tab10.colors
    
    for i, instrument in enumerate(midi.instruments):
        if instrument.is_drum:
            continue
        color = colors[i % len(colors)]
        for note in instrument.notes:
            if note.start > max_time:
                break
            end = min(note.end, max_time)
            rect = plt.Rectangle(
                (note.start, note.pitch - 0.4),
                end - note.start,
                0.8,
                color=color,
                alpha=0.7,
            )
            ax.add_patch(rect)
    
    ax.set_xlim(0, max_time)
    ax.set_ylim(20, 100)
    ax.set_xlabel("Tiempo (segundos)")
    ax.set_ylabel("Nota MIDI (60 = C4)")
    ax.set_title(title)
    
    # Marcar octavas
    for octave_c in range(24, 108, 12):
        ax.axhline(y=octave_c, color='gray', alpha=0.3, linestyle='--', linewidth=0.5)
        ax.text(-0.3, octave_c, f"C{octave_c//12 - 1}", fontsize=7, va='center')
    
    plt.tight_layout()
    plt.show()


# Mostrar el piano roll del primer archivo cargado
if midis:
    plot_piano_roll(midis[0], f"Piano Roll: {loaded_names[0]} (primeros 30s)")
else:
    print("No hay archivos cargados para visualizar.")

## 3. Pitch Class Histogram — el feature de entrada para key detection

El histograma de pitch class colapsa el piano roll en 12 números: 
cuánto tiempo estuvo activa cada nota (Do, Do#, Re, ..., Si), 
independientemente de la octava.

**¿Qué deberíamos ver?**
- Si la pieza está en Do mayor: picos en 0 (C), 4 (E), 7 (G)
- Si está en La menor: picos en 9 (A), 0 (C), 4 (E)
- Si es atonal o muy cromática: distribución más plana

La **forma** del histograma es la firma tonal de la pieza. 
Es lo que el modelo aprende a clasificar.

In [ ]:
PITCH_CLASS_NAMES = ["C", "C#", "D", "D#", "E", "F", "F#", "G", "G#", "A", "A#", "B"]

def plot_pitch_class_histogram(midi: pretty_midi.PrettyMIDI, title: str):
    """Visualiza el histograma de pitch class de un archivo MIDI."""
    hist = get_pitch_class_histogram(midi, normalize=True)
    
    fig, ax = plt.subplots(figsize=(10, 4))
    bars = ax.bar(range(12), hist, color=plt.cm.RdYlGn(hist / hist.max() if hist.max() > 0 else hist))
    ax.set_xticks(range(12))
    ax.set_xticklabels(PITCH_CLASS_NAMES)
    ax.set_ylabel("Fracción del tiempo activa (suma = 1.0)")
    ax.set_title(title)
    ax.set_ylim(0, 0.25)
    
    # Mostrar valor sobre cada barra
    for i, (bar, val) in enumerate(zip(bars, hist)):
        if val > 0.01:
            ax.text(bar.get_x() + bar.get_width()/2, val + 0.003, 
                    f"{val:.2f}", ha='center', va='bottom', fontsize=8)
    
    plt.tight_layout()
    plt.show()
    
    # Reportar el pitch class dominante
    dominant_pc = np.argmax(hist)
    print(f"Nota dominante: {PITCH_CLASS_NAMES[dominant_pc]} ({hist[dominant_pc]:.1%} del tiempo)")
    print(f"Suma del histograma: {hist.sum():.10f}")
    return hist


if midis:
    for midi, name in zip(midis[:3], loaded_names[:3]):
        plot_pitch_class_histogram(midi, f"Pitch Class Histogram: {name}")

## 4. Distribución agregada — ¿qué tonalidades son más comunes en el dataset?

Esto responde una pregunta crucial: **¿el dataset está balanceado entre tonalidades?**

Si el 60% de los archivos están en Do mayor y los otros 23 tonalidades se reparten 
el 40%, el modelo aprenderá a predecir "Do mayor" para todo y tendrá buen accuracy 
en el paper pero mal en uso real. Este sesgo de clase es uno de los errores más 
comunes en datasets de música occidental.

Mitigación: data augmentation por transposición (transponer cada archivo a las 
12 tonalidades) balancea artificialmente el dataset.

In [ ]:
# Calcular el histograma promedio sobre todos los archivos cargados
if midis:
    all_histograms = np.array([get_pitch_class_histogram(m, normalize=True) for m in midis])
    mean_histogram = all_histograms.mean(axis=0)
    std_histogram = all_histograms.std(axis=0)
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 4))
    
    # Histograma promedio
    ax = axes[0]
    ax.bar(range(12), mean_histogram, yerr=std_histogram, capsize=3,
           color='steelblue', alpha=0.8, label='Media')
    ax.set_xticks(range(12))
    ax.set_xticklabels(PITCH_CLASS_NAMES)
    ax.set_title(f"Histograma promedio ({len(midis)} archivos)")
    ax.set_ylabel("Fracción del tiempo activa")
    ax.legend()
    
    # Heatmap: cada fila es un archivo MIDI, cada columna un pitch class
    ax = axes[1]
    im = ax.imshow(all_histograms, aspect='auto', cmap='YlOrRd', vmin=0, vmax=0.2)
    ax.set_xticks(range(12))
    ax.set_xticklabels(PITCH_CLASS_NAMES)
    ax.set_yticks(range(len(loaded_names)))
    ax.set_yticklabels([n[:30] for n in loaded_names], fontsize=8)
    ax.set_title("Heatmap: pitch class por archivo")
    plt.colorbar(im, ax=ax, label="Fraccion activa")
    
    plt.tight_layout()
    plt.show()
    
    print("\nPitch class mas activo por archivo:")
    for name, hist in zip(loaded_names, all_histograms):
        dominant = np.argmax(hist)
        print(f"  {name[:40]:<40} → {PITCH_CLASS_NAMES[dominant]} ({hist[dominant]:.1%})")

## 5. Estadísticas de densidad

**Densidad** = notas por segundo. Es una característica importante del dataset porque:

- Determina el tamaño de la ventana temporal óptima para el modelo.
  Si la música promedia 4 notas/segundo, una ventana de 2 segundos captura ~8 notas — 
  suficiente para estimar la tonalidad.
  
- Indica si el dataset es representativo del uso real del Brain.
  Si el usuario toca a 1 nota/segundo (lento) pero el dataset tiene 10 notas/segundo 
  (jazz rápido), el modelo puede no generalizar bien.

**Rango tonal** = diferencia entre la nota más alta y la más baja. 
Indica si la pieza usa un rango amplio (piano clásico) o restringido (melodía simple).

In [ ]:
def midi_stats(midi: pretty_midi.PrettyMIDI) -> dict:
    """Calcula estadísticas básicas de un archivo MIDI."""
    seq = midi_to_note_sequence(midi)
    if not seq:
        return {"total_notes": 0, "duration_s": 0}
    
    duration = midi.get_end_time()
    pitches = [n["pitch"] for n in seq]
    velocities = [n["velocity"] for n in seq]
    note_durations = [n["duration"] for n in seq]
    
    return {
        "total_notes": len(seq),
        "duration_s": round(duration, 1),
        "notes_per_second": round(len(seq) / duration, 1) if duration > 0 else 0,
        "pitch_range": max(pitches) - min(pitches),
        "pitch_min": min(pitches),
        "pitch_max": max(pitches),
        "velocity_mean": round(np.mean(velocities), 1),
        "note_duration_mean_ms": round(np.mean(note_durations) * 1000, 1),
        "n_instruments": len([i for i in midi.instruments if not i.is_drum]),
        "has_drums": any(i.is_drum for i in midi.instruments),
    }


if midis:
    print(f"{'Archivo':<35} {'Notas':>6} {'Dur(s)':>7} {'N/s':>5} {'Rango':>6} {'Vel':>5} {'NoteDur(ms)':>12}")
    print("-" * 80)
    
    all_stats = []
    for midi, name in zip(midis, loaded_names):
        stats = midi_stats(midi)
        all_stats.append(stats)
        print(
            f"{name[:35]:<35} "
            f"{stats['total_notes']:>6} "
            f"{stats['duration_s']:>7.1f} "
            f"{stats['notes_per_second']:>5.1f} "
            f"{stats['pitch_range']:>6} "
            f"{stats['velocity_mean']:>5.0f} "
            f"{stats['note_duration_mean_ms']:>12.0f}"
        )
    
    if len(all_stats) > 1:
        print("-" * 80)
        nps_vals = [s['notes_per_second'] for s in all_stats if s['duration_s'] > 0]
        print(f"\nNotas/segundo — media: {np.mean(nps_vals):.1f}, mediana: {np.median(nps_vals):.1f}, max: {max(nps_vals):.1f}")
        print()
        print("Implicacion para key detection:")
        median_nps = np.median(nps_vals)
        window_for_10_notes = 10 / median_nps if median_nps > 0 else float('inf')
        print(f"  Con densidad mediana de {median_nps:.1f} notas/s, necesitamos ~{window_for_10_notes:.1f}s para acumular 10 notas.")
        print(f"  El modelo de key detection puede usar una ventana de {max(1, int(window_for_10_notes))}-{max(2, int(window_for_10_notes)*2)}s.")

## 6. Data augmentation por transposición

El dataset de Lakh tiene una distribución sesgada de tonalidades — 
la música pop/rock occidental tiene mucho C mayor y G mayor.

La solución estándar es **transponer** cada archivo a las 12 tonalidades posibles.
Un archivo en Do mayor, transpuesto +1 semitono, se convierte en Re bemol mayor. 
+2 → Re mayor. Etc. Esto multiplica el dataset por 12x sin nueva grabación.

**¿Cambia algo importante con la transposición?**
- Los pitches cambian. C4 → D4 con +2 semitonos.
- Las duraciones NO cambian. El timing musical es invariante de tonalidad.
- El pitch class histogram rota. Si C era dominante en el original,
  D será dominante en el transposed +2.

Esto nos permite ver visualmente que la transposición "rota" el histograma.

In [ ]:
from utils.midi_utils import transpose_midi

if midis:
    midi_0 = midis[0]
    
    fig, axes = plt.subplots(2, 6, figsize=(18, 6))
    axes = axes.flatten()
    
    for i in range(12):
        ax = axes[i]
        transposed = transpose_midi(midi_0, semitones=i)
        hist = get_pitch_class_histogram(transposed, normalize=True)
        ax.bar(range(12), hist, color='steelblue', alpha=0.8)
        ax.set_xticks(range(12))
        ax.set_xticklabels(PITCH_CLASS_NAMES, fontsize=7)
        ax.set_title(f"+{i} semitono{'s' if i != 1 else ''}", fontsize=9)
        ax.set_ylim(0, 0.30)
        ax.tick_params(axis='y', labelsize=7)
    
    plt.suptitle(
        f"Data Augmentation: {loaded_names[0]} en las 12 tonalidades\n"
        "(el histograma rota — misma forma, distintas notas dominantes)",
        fontsize=11
    )
    plt.tight_layout()
    plt.show()
    
    print("Verificacion: la nota dominante rota con la transposicion:")
    dominant_original = np.argmax(get_pitch_class_histogram(midi_0))
    for i in range(12):
        transposed = transpose_midi(midi_0, semitones=i)
        hist = get_pitch_class_histogram(transposed)
        dominant = np.argmax(hist)
        expected = (dominant_original + i) % 12
        status = "OK" if dominant == expected else f"WARN (esperado {PITCH_CLASS_NAMES[expected]})"
        print(f"  +{i:2d} semitones → dominante: {PITCH_CLASS_NAMES[dominant]:3s}  [{status}]")

## 7. Inferencia de tonalidad con music21 (ground truth para training)

Los archivos MIDI de Lakh no tienen la tonalidad anotada. Necesitamos inferirla 
algorítmicamente para generar las etiquetas de entrenamiento.

Usamos `music21.analysis.discrete.KrumhanslSchmuckler()` — implementación del 
algoritmo de correlación de perfiles tonales de Krumhansl & Schmuckler (1990).

El algoritmo:
1. Calcula el pitch class histogram del archivo
2. Lo correlaciona con los 24 perfiles teóricos (12 mayor + 12 menor)
3. La tonalidad con mayor correlación es el output

Accuracy del algoritmo: ~90% en música tonal occidental. Suficiente para training.

In [ ]:
try:
    import music21
    from music21 import converter, analysis
    MUSIC21_AVAILABLE = True
except ImportError:
    MUSIC21_AVAILABLE = False
    print("music21 no disponible — saltar análisis de tonalidad")

def infer_key_music21(midi_path: Path) -> str | None:
    """Infiere la tonalidad de un MIDI usando el algoritmo de Krumhansl-Schmuckler."""
    if not MUSIC21_AVAILABLE:
        return None
    try:
        score = converter.parse(str(midi_path))
        key = score.analyze('key')
        return str(key)  # Ej: "C major", "a minor"
    except Exception as e:
        return f"Error: {e}"


if MUSIC21_AVAILABLE and midi_files:
    print("Analizando tonalidades (puede tardar 10-30s por archivo)...")
    print()
    for path in midi_files[:5]:  # solo los primeros 5 para no tardar mucho
        key = infer_key_music21(path)
        hist = get_pitch_class_histogram(load_midi(path))
        dominant_pc = PITCH_CLASS_NAMES[np.argmax(hist)]
        print(f"  {path.name[:40]:<40} → Tonalidad: {key:<15} (nota dominante: {dominant_pc})")
    
    print()
    print("La tonalidad inferida y la nota dominante deberían concordar en ~90% de los casos.")
    print("Discordancias son normales (modo menor vs mayor con misma nota raíz, etc.)")
else:
    print("music21 no disponible o no hay archivos. Instalar con: uv add music21")

## 8. Resumen — decisiones de diseño del modelo

Con lo que vimos en este notebook, podemos informar las decisiones de arquitectura 
para el Sprint 4.3 (Scale Lock model):

**Feature de entrada confirmada:** pitch class histogram (12 dimensiones)
- Compacto, invariante de octava, invariante de tempo
- Suma = 1.0 → el modelo no necesita normalizar su entrada
- Capta la información relevante para key detection

**Data augmentation:** transposición a 12 tonalidades × dataset size
- Balancea la distribución sesgada del dataset
- Multiplica 12x los datos de entrenamiento sin nueva recolección

**Tamaño de la arquitectura:**
- Input: 12 features
- Output: 24 clases (12 mayor + 12 menor)
- Con input pequeño, una red Dense de 2-3 capas es suficiente
- Target: 12KB en INT8 → ~12,000 parámetros → totalmente viable

**Ground truth para etiquetas:**
- `music21` con Krumhansl-Schmuckler (~90% accuracy)
- Ruido de etiquetado del 10% es aceptable para nuestro caso de uso

**Próximo sprint (4.3):** escala desde aquí — dataset, training, quantización, export a `.h`.

In [ ]:
# Verificación final: la pipeline completa funciona
from utils.tflite_utils import tflite_to_c_array
import tempfile

print("Verificacion del pipeline completo:")
print()

# 1. Cargar MIDI
if midi_files:
    midi = load_midi(midi_files[0])
    print(f"  1. load_midi()                  OK — {len(midi.instruments)} instrumentos")
    
    # 2. Feature extraction
    hist = get_pitch_class_histogram(midi)
    assert abs(hist.sum() - 1.0) < 1e-9 or hist.sum() == 0, "histograma no suma 1"
    print(f"  2. get_pitch_class_histogram()  OK — shape {hist.shape}, suma={hist.sum():.6f}")
    
    # 3. Data augmentation
    transposed = transpose_midi(midi, semitones=6)
    print(f"  3. transpose_midi(+6)           OK — {sum(len(i.notes) for i in transposed.instruments)} notas")
    
    # 4. C array generation (con datos dummy)
    with tempfile.NamedTemporaryFile(suffix='.tflite', delete=False) as tmp:
        tmp.write(b'\x1c\x00\x00\x00TFL3' + b'\x00' * 100)  # header TFLite mínimo
        tmp_path = Path(tmp.name)
    
    c_header = tflite_to_c_array(tmp_path, 'g_dummy_model')
    assert 'alignas(8)' in c_header
    assert 'g_dummy_model_len' in c_header
    assert '#pragma once' in c_header
    print(f"  4. tflite_to_c_array()          OK — {len(c_header)} chars generados")
    tmp_path.unlink()
    
    print()
    print("Pipeline completo: OK")
    print("Listo para Sprint 4.3 — Scale Lock model")
else:
    print("  No hay archivos MIDI disponibles para verificacion.")
    print("  Correr: uv run python datasets/download_lakh.py")